# 03 (versão 3) — CNN 2D com passo 2 na primeira convolução

Mesma preparação, normalização e protocolo da versão 2 (linha de base por máquina, cinco sementes, int8). Muda **uma coisa**:

* **Passo (stride) 2 na primeira convolução.** Na placa, a CNN v2 levou cerca de 363 ms por janela (Harvard_TinyMLx, 64 MHz), acima do limite de 200 ms do RNF2. Com passo 2, a saída da primeira camada cai de 98 × 40 para 49 × 20, e todas as camadas seguintes processam 4 vezes menos posições: de cerca de 2,5 para 0,62 milhões de multiplicações por janela. O número de parâmetros não muda (6.450).
* **Quantização já no formato embarcado:** int8 com as camadas densas quantizadas **por tensor** (compatível com a Harvard_TinyMLx).
* **Comparação com a v2:** desempenho no teste (5 sementes) e o teste de generalização "deixar uma máquina de fora", rodado aqui para as **duas** arquiteturas com o mesmo procedimento.

Saídas em `modelos/cnn_v3/` (a v2 continua em `modelos/cnn_v2/`). Depois deste notebook, rode o **05** com `VERSAO_CNN = "cnn_v3"`.

**Ambiente:** GPU T4. Tempo estimado: 1 a 2 horas (o teste de generalização treina 8 modelos).

In [ ]:
# 1. Ambiente, versões e dados
import sys, os, json, platform, subprocess, importlib.util
if importlib.util.find_spec("tensorflow") is None:          # algumas imagens do Colab vêm sem TensorFlow
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow==2.21.0"], check=True)
import numpy as np, tensorflow as tf
print("Python", sys.version.split()[0], "| TensorFlow", tf.__version__, "| Keras", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "")
print("GPU:", tf.config.list_physical_devices("GPU"))
from google.colab import drive
drive.mount("/content/drive")

DADOS = "/content/drive/MyDrive/dissertacao_tinyml/mimii_fan_6db"
SAIDA = "/content/drive/MyDrive/dissertacao_tinyml/modelos/cnn_v3"
os.makedirs(SAIDA, exist_ok=True)
import pandas as pd
MAQ = pd.read_csv(f"{DADOS}/particao_por_gravacao.csv")["maquina"].values   # máquina de cada gravação (índice = grupo)

def bruto(p):
    d = np.load(f"{DADOS}/{p}.npz")
    return d["X"].astype(np.float32), d["y"].astype(np.int32), d["grupo"]
Xtr, ytr, gtr = bruto("treino"); Xva, yva, gva = bruto("validacao"); Xte, yte, gte = bruto("teste")

# Linha de base de cada máquina: só janelas NORMAIS do TREINO (sem vazamento para validação/teste)
STATS = {}
for m in sorted(set(MAQ)):
    ref = (MAQ[gtr] == m) & (ytr == 0)
    STATS[m] = (Xtr[ref].mean(axis=(0, 1)), Xtr[ref].std(axis=(0, 1)) + 1e-6)

def normalizar(X, g):
    out = np.empty_like(X)
    for m, (mu, sd) in STATS.items():
        s = MAQ[g] == m
        out[s] = (X[s] - mu) / sd
    return out[..., None]                                   # (n, 98, 40, 1)
Xtr, Xva, Xte = normalizar(Xtr, gtr), normalizar(Xva, gva), normalizar(Xte, gte)
print("Linha de base calculada para:", list(STATS))
for n, y in [("treino", ytr), ("validação", yva), ("teste", yte)]:
    print(f"{n:10s} n={len(y):6d}  normal={int((y==0).sum())}  anômalo={int((y==1).sum())}")

In [ ]:
# 2. Modelo e configuração de treino
HP = {"filtros": [8, 16, 32], "densa": 16, "lr": 1e-3, "lote": 32, "epocas": 100, "paciencia": 10, "passo1": 2}
SEMENTES = [0, 1, 2, 3, 4]

def criar_cnn(n_classes=2, filtros=HP["filtros"], densa=HP["densa"], passo1=HP["passo1"]):
    ent = tf.keras.Input(shape=Xtr.shape[1:], name="logmel")
    x = ent
    for i, f in enumerate(filtros):
        x = tf.keras.layers.Conv2D(f, 3, strides=passo1 if i == 0 else 1, padding="same", activation="relu",
                                   name=f"conv{i+1}")(x)
        if i < len(filtros) - 1:
            x = tf.keras.layers.MaxPooling2D(2, name=f"pool{i+1}")(x)
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = tf.keras.layers.Dense(densa, activation="relu", name="densa")(x)
    sai = tf.keras.layers.Dense(n_classes, activation="softmax", name="saida")(x)
    return tf.keras.Model(ent, sai, name="cnn2d")

# Pesos pelo inverso da frequência das classes no treino
cont = np.bincount(ytr); PESOS = {c: len(ytr) / (len(cont) * n) for c, n in enumerate(cont)}
print("Pesos das classes:", {k: round(v, 3) for k, v in PESOS.items()})
criar_cnn().summary()
def macs(m):
    # multiplicações por janela nas convoluções e densas (comparação com a v2)
    t = 0
    for l in m.layers:
        if isinstance(l, tf.keras.layers.Conv2D):
            k = l.kernel.shape; o = l.output.shape
            t += int(o[1] * o[2] * o[3] * k[0] * k[1] * k[2])
        elif isinstance(l, tf.keras.layers.Dense):
            t += int(l.kernel.shape[0] * l.kernel.shape[1])
    return t
print(f"Multiplicações por janela: v3 {macs(criar_cnn()):,} | v2 {macs(criar_cnn(passo1=1)):,}".replace(",", "."))

In [ ]:
# 3. Treino com cinco sementes
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, roc_auc_score, confusion_matrix
import pandas as pd

def metricas(y, prob):
    pred = prob.argmax(1)
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=[0, 1], zero_division=0)
    return {"acuracia": accuracy_score(y, pred), "f1_macro": f1_score(y, pred, average="macro"),
            "precisao_normal": p[0], "recall_normal": r[0], "f1_normal": f[0],
            "precisao_anomalo": p[1], "recall_anomalo": r[1], "f1_anomalo": f[1],
            "auc": roc_auc_score(y, prob[:, 1])}

def por_clipe(prob, grupo, y):
    """Agregação por clipe de 10 s: média das probabilidades das janelas."""
    df = pd.DataFrame({"g": grupo, "p": prob[:, 1], "y": y}).groupby("g").mean()
    return np.stack([1 - df.p, df.p], 1), df.y.astype(int).values

resultados, historicos = [], {}
for s in SEMENTES:
    tf.keras.utils.set_random_seed(s)
    modelo = criar_cnn()
    modelo.compile(optimizer=tf.keras.optimizers.Adam(HP["lr"]), loss="sparse_categorical_crossentropy",
                   metrics=["accuracy"])
    parada = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=HP["paciencia"], restore_best_weights=True)
    h = modelo.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=HP["epocas"], batch_size=HP["lote"],
                   class_weight=PESOS, callbacks=[parada], verbose=2)
    historicos[s] = h.history
    prob_va, prob_te = modelo.predict(Xva, verbose=0), modelo.predict(Xte, verbose=0)
    m = {"semente": s, "epocas": len(h.history["loss"]), "f1_macro_validacao": f1_score(yva, prob_va.argmax(1), average="macro")}
    m.update({f"teste_{k}": v for k, v in metricas(yte, prob_te).items()})
    pc, yc = por_clipe(prob_te, gte, yte)
    m.update({f"clipe_{k}": v for k, v in metricas(yc, pc).items()})
    resultados.append(m)
    modelo.save(f"{SAIDA}/cnn_semente{s}.keras")
    print(f"semente {s}: F1 macro (teste) = {m['teste_f1_macro']:.4f} | AUC = {m['teste_auc']:.4f}")

res = pd.DataFrame(resultados).set_index("semente")
res.to_csv(f"{SAIDA}/resultados_sementes.csv")
resumo = res.agg(["mean", "std"]).T
display(resumo.round(4)); resumo.to_csv(f"{SAIDA}/resultados_media_desvio.csv")

In [ ]:
# 4. Curvas de treino e matriz de confusão do modelo escolhido
import matplotlib.pyplot as plt
ESCOLHIDA = int(res["f1_macro_validacao"].idxmax())      # escolha pela validação, nunca pelo teste
print("Semente escolhida para implantação (maior F1 macro na validação):", ESCOLHIDA)
modelo = tf.keras.models.load_model(f"{SAIDA}/cnn_semente{ESCOLHIDA}.keras")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for s, h in historicos.items():
    ax[0].plot(h["loss"], lw=.8, label=f"treino s{s}"); ax[0].plot(h["val_loss"], "--", lw=.8)
    ax[1].plot(h["val_accuracy"], lw=.8, label=f"s{s}")
ax[0].set_title("Perda (contínua: treino; tracejada: validação)"); ax[1].set_title("Acurácia na validação")
ax[1].legend(fontsize=7); plt.tight_layout(); plt.savefig(f"{SAIDA}/curvas_treino.png", dpi=150); plt.show()

cm = confusion_matrix(yte, modelo.predict(Xte, verbose=0).argmax(1))
print("Matriz de confusão (teste, por janela) — linhas: real; colunas: predito [normal, anômalo]"); print(cm)

In [ ]:
# 5. Conversão para TFLite: float32 e int8 (entradas e saídas; densas por tensor), calibração com 100 amostras do treino
rng = np.random.default_rng(0)
idx_rep = rng.choice(len(Xtr), 100, replace=False)
def dataset_representativo():
    for i in idx_rep:
        yield [Xtr[i:i+1]]

conv = tf.lite.TFLiteConverter.from_keras_model(modelo)
tfl_f32 = conv.convert()

conv = tf.lite.TFLiteConverter.from_keras_model(modelo)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = dataset_representativo
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8
conv.inference_output_type = tf.int8
conv._experimental_disable_per_channel_quantization_for_dense_layers = True   # densas por tensor (Harvard_TinyMLx)
tfl_i8 = conv.convert()

open(f"{SAIDA}/cnn_float32.tflite", "wb").write(tfl_f32)
open(f"{SAIDA}/cnn_int8.tflite", "wb").write(tfl_i8)
kb = lambda b: len(b) / 1024
print(f"float32: {kb(tfl_f32):.1f} KB | int8: {kb(tfl_i8):.1f} KB | redução: {100*(1-len(tfl_i8)/len(tfl_f32)):.1f}%")
print("Parâmetros treináveis:", modelo.count_params())

In [ ]:
# 6. Avaliação da versão int8 contra a float32 (mesmo conjunto de teste)
try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    Interpreter = tf.lite.Interpreter

def prever_tflite(modelo_bytes, X):
    it = Interpreter(model_content=modelo_bytes); it.allocate_tensors()
    ent, sai = it.get_input_details()[0], it.get_output_details()[0]
    esc_e, zero_e = ent["quantization"]; esc_s, zero_s = sai["quantization"]
    out = np.zeros((len(X), sai["shape"][-1]), np.float32)
    for i in range(len(X)):
        x = X[i:i+1]
        if ent["dtype"] == np.int8:
            x = np.clip(np.round(x / esc_e + zero_e), -128, 127).astype(np.int8)
        it.set_tensor(ent["index"], x); it.invoke()
        y = it.get_tensor(sai["index"]).astype(np.float32)
        out[i] = (y - zero_s) * esc_s if sai["dtype"] == np.int8 else y
    return out

p32 = prever_tflite(tfl_f32, Xte); p8 = prever_tflite(tfl_i8, Xte)
comp = pd.DataFrame({"float32": metricas(yte, p32), "int8": metricas(yte, p8)})
comp["diferença"] = comp["int8"] - comp["float32"]
display(comp.round(4)); comp.to_csv(f"{SAIDA}/float32_vs_int8.csv")
print("Concordância de predições float32 x int8:", f"{100*(p32.argmax(1)==p8.argmax(1)).mean():.2f}%")

it = Interpreter(model_content=tfl_i8); it.allocate_tensors()
q = {"entrada": [float(v) for v in it.get_input_details()[0]["quantization"]],
     "saida": [float(v) for v in it.get_output_details()[0]["quantization"]]}
json.dump({"semente": ESCOLHIDA, "quantizacao": q, "normalizacao": "linha de base por máquina e por filtro mel",
           "linhas_de_base_mimii": {m: {"media": mu.tolist(), "desvio": sd.tolist()} for m, (mu, sd) in STATS.items()}},
          open(f"{SAIDA}/cnn_int8_quantizacao.json", "w"), indent=2)
print("Escala/zero de entrada e saída (o firmware precisa deles):", q)

In [ ]:
# 7. Comparação com a v2 (cinco sementes, mesmo teste)
V2 = SAIDA.replace("cnn_v3", "cnn_v2")
cols = ["teste_acuracia", "teste_f1_macro", "teste_auc", "clipe_acuracia", "clipe_f1_macro"]
r2 = pd.read_csv(f"{V2}/resultados_sementes.csv", index_col=0)[cols]
r3 = res[cols]
fmt = lambda r: r.mean().map("{:.4f}".format) + " ± " + r.std().map("{:.4f}".format)
tab = pd.DataFrame({"v2 (passo 1)": fmt(r2), "v3 (passo 2)": fmt(r3)})
tab.loc["multiplicações por janela"] = [f"{macs(criar_cnn(passo1=1)):,}".replace(",", "."), f"{macs(modelo):,}".replace(",", ".")]
tab.loc["int8 (KB)"] = [f"{os.path.getsize(f'{V2}/cnn_int8.tflite')/1024:.1f}", f"{len(tfl_i8)/1024:.1f}"]
display(tab); tab.to_csv(f"{SAIDA}/comparacao_v2_v3.csv")

In [ ]:
# 8. Generalização: deixar uma máquina de fora (v2 e v3, mesmo procedimento)
# Para cada máquina m: treina com as outras três (treino e validação delas, linha de base de cada uma);
# em m, 20% das gravações normais formam a linha de base e saem do teste; o teste usa as demais gravações de m.
# Uma semente (0) por modelo, para caber no tempo do Colab.
def bruto_todos():
    Xs, ys, gs, ps = [], [], [], []
    for p in ["treino", "validacao", "teste"]:
        X, y, g = bruto(p); Xs.append(X); ys.append(y); gs.append(g); ps += [p] * len(y)
    return np.concatenate(Xs), np.concatenate(ys), np.concatenate(gs), np.array(ps)
XA, yA, gA, pA = bruto_todos()
mA = MAQ[gA]
rng = np.random.default_rng(42)
auc = {}
for m in sorted(set(MAQ)):
    stats = {}
    for o in sorted(set(MAQ) - {m}):
        ref = (mA == o) & (pA == "treino") & (yA == 0)
        stats[o] = (XA[ref].mean(axis=(0, 1)), XA[ref].std(axis=(0, 1)) + 1e-6)
    grav_norm = np.unique(gA[(mA == m) & (yA == 0)])
    base_g = rng.choice(grav_norm, int(round(0.2 * len(grav_norm))), replace=False)
    ref = (mA == m) & np.isin(gA, base_g)
    stats[m] = (XA[ref].mean(axis=(0, 1)), XA[ref].std(axis=(0, 1)) + 1e-6)
    def norm(sel):
        X = np.empty((sel.sum(),) + XA.shape[1:], np.float32)
        for o, (mu, sd) in stats.items():
            s = mA[sel] == o
            X[s] = (XA[sel][s] - mu) / sd
        return X[..., None]
    tr = (mA != m) & (pA == "treino"); va = (mA != m) & (pA == "validacao"); te = (mA == m) & ~np.isin(gA, base_g)
    Xtr_m, Xva_m, Xte_m = norm(tr), norm(va), norm(te)
    cont_m = np.bincount(yA[tr]); pesos_m = {c: len(yA[tr]) / (len(cont_m) * n) for c, n in enumerate(cont_m)}
    for nome, p1 in [("v2 (passo 1)", 1), ("v3 (passo 2)", 2)]:
        tf.keras.utils.set_random_seed(0)
        mdl = criar_cnn(passo1=p1)
        mdl.compile(optimizer=tf.keras.optimizers.Adam(HP["lr"]), loss="sparse_categorical_crossentropy")
        mdl.fit(Xtr_m, yA[tr], validation_data=(Xva_m, yA[va]), epochs=HP["epocas"], batch_size=HP["lote"],
                class_weight=pesos_m, verbose=0,
                callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=HP["paciencia"], restore_best_weights=True)])
        auc[(m, nome)] = roc_auc_score(yA[te], mdl.predict(Xte_m, verbose=0)[:, 1])
        print(f"{m} | {nome}: AUC {auc[(m, nome)]:.3f}")
    del Xtr_m, Xva_m, Xte_m
ger = pd.Series(auc).unstack()
ger.loc["média"] = ger.mean()
display(ger.round(3)); ger.to_csv(f"{SAIDA}/generalizacao_maquina_nova.csv")

## Resultado

Em `dissertacao_tinyml/modelos/cnn_v3/`:

| Arquivo | Uso na dissertação |
|---|---|
| `resultados_sementes.csv`, `resultados_media_desvio.csv` | seção 8 (pré-treinamento): média ± desvio em cinco sementes, por janela e por clipe |
| `curvas_treino.png` | figura de convergência |
| `cnn_float32.tflite`, `cnn_int8.tflite` | Tabela 10 (tamanho antes e após a quantização) |
| `float32_vs_int8.csv` | comparação int8 × float32 (Apêndice B.7) |
| `cnn_int8_quantizacao.json` | escalas e linhas de base (lidas pelo notebook 05) |
| `comparacao_v2_v3.csv` | v2 × v3: desempenho, multiplicações e tamanho |
| `generalizacao_maquina_nova.csv` | AUC "deixar uma máquina de fora" (v2 e v3) |

**Próximo passo:** notebook 05 com `VERSAO_CNN = "cnn_v3"` (gera os `.h`).

As métricas são do **pré-treinamento binário no MIMII**; elas **não** testam H1 nem H2, que dependem da coleta in situ.